In [ ]:
!pip install imbalanced-learn xgboost lightgbm plotly -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             f1_score, precision_score, recall_score, accuracy_score)

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

import xgboost as xgb
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

print(" All libraries imported successfully!")

In [ ]:
from google.colab import files
files.upload()

In [ ]:
df = pd.read_csv('creditcard.csv')
df.head()

In [ ]:
def perform_eda(df):
    print("="*60)
    print(" EXPLORATORY DATA ANALYSIS")
    print("="*60)

    print("\n Dataset Overview:")
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nMissing values:\n{df.isnull().sum().sum()} total")

    print("\n Class Distribution:")
    fraud_counts = df['Class'].value_counts()
    print(fraud_counts)
    fraud_pct = (fraud_counts[1] / len(df)) * 100
    print(f"\nFraud percentage: {fraud_pct:.4f}%")

    print("\n Statistical Summary:")
    print(df.describe())

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Class Distribution', 'Transaction Amount Distribution',
                       'Time Distribution', 'Amount by Class'),
        specs=[[{'type': 'bar'}, {'type': 'histogram'}],
               [{'type': 'histogram'}, {'type': 'box'}]]
    )

    fig.add_trace(
        go.Bar(x=['Normal', 'Fraud'], y=fraud_counts.values,
               marker_color=['lightblue', 'red']),
        row=1, col=1
    )

    fig.add_trace(
        go.Histogram(x=df['Amount'], nbinsx=50, name='Amount'),
        row=1, col=2
    )

    fig.add_trace(
        go.Histogram(x=df['Time'], nbinsx=50, name='Time'),
        row=2, col=1
    )

    fig.add_trace(
        go.Box(y=df[df['Class']==0]['Amount'], name='Normal'),
        row=2, col=2
    )
    fig.add_trace(
        go.Box(y=df[df['Class']==1]['Amount'], name='Fraud'),
        row=2, col=2
    )

    fig.update_layout(height=800, showlegend=False, title_text="EDA Overview")
    fig.show()

    print("\n Correlation Analysis:")
    corr_features = ['Amount', 'Class'] + [col for col in df.columns if col.startswith('V')][:5]
    corr_matrix = df[corr_features].corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title('Correlation Matrix')
    plt.tight_layout()
    plt.show()

perform_eda(df)

In [ ]:
def preprocess_data(df, test_size=0.3, random_state=42):
    """Preprocess the data for modeling"""

    print("="*60)
    print(" DATA PREPROCESSING")
    print("="*60)

    df_cleaned = df.dropna(subset=['Class'])
    print(f"\n Dropped {len(df) - len(df_cleaned)} rows with NaN values in 'Class' column.")

    X = df_cleaned.drop('Class', axis=1)
    y = df_cleaned['Class']

    print(f"\n Features shape: {X.shape}")
    print(f"Target shape: {y.shape}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    print(f"\n Training set: {X_train.shape}")
    print(f" Test set: {X_test.shape}")
    print(f"\nTraining fraud cases: {y_train.sum()}")
    print(f"Test fraud cases: {y_test.sum()}")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print("\n Features scaled successfully!")

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler


X_train, X_test, y_train, y_test, scaler = preprocess_data(df)

In [ ]:
def apply_sampling(X_train, y_train, method='smote'):
    """Apply sampling techniques to handle imbalance"""

    print("="*60)
    print("HANDLING CLASS IMBALANCE")
    print("="*60)

    print(f"\nOriginal class distribution:")
    print(pd.Series(y_train).value_counts())

    if method == 'smote':
        sampler = SMOTE(random_state=42)
        X_resampled, y_resampled = sampler.fit_resample(X_train, y_train)
        print(f"\n Applied SMOTE")

    elif method == 'undersample':
        sampler = RandomUnderSampler(random_state=42)
        X_resampled, y_resampled = sampler.fit_resample(X_train, y_train)
        print(f"\n Applied Random Undersampling")

    elif method == 'combined':
        over = SMOTE(sampling_strategy=0.1, random_state=42)
        under = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
        X_resampled, y_resampled = over.fit_resample(X_train, y_train)
        X_resampled, y_resampled = under.fit_resample(X_resampled, y_resampled)
        print(f"\n Applied Combined Sampling")

    print(f"\nResampled class distribution:")
    print(pd.Series(y_resampled).value_counts())

    return X_resampled, y_resampled

X_train_resampled, y_train_resampled = apply_sampling(X_train, y_train, method='smote')


In [ ]:
def train_models(X_train, y_train, X_test, y_test):
    """Train multiple models and compare performance"""

    print("="*60)
    print("MODEL TRAINING")
    print("="*60)

    models = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
        'LightGBM': lgb.LGBMClassifier(random_state=42, verbose=-1)
    }

    results = {}

    for name, model in models.items():
        print(f"\n{'='*40}")
        print(f"Training {name}...")
        print(f"{'='*40}")

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_pred_proba)

        results[name] = {
            'model': model,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'roc_auc': roc_auc,
            'predictions': y_pred,
            'probabilities': y_pred_proba
        }

        print(f"Accuracy:  {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1 Score:  {f1:.4f}")
        print(f"ROC AUC:   {roc_auc:.4f}")

    return results

results = train_models(X_train_resampled, y_train_resampled, X_test, y_test)

In [ ]:
def evaluate_models(results, y_test):
    """Comprehensive model evaluation"""

    print("="*60)
    print("MODEL EVALUATION")
    print("="*60)

    comparison = pd.DataFrame({
        'Model': list(results.keys()),
        'Accuracy': [r['accuracy'] for r in results.values()],
        'Precision': [r['precision'] for r in results.values()],
        'Recall': [r['recall'] for r in results.values()],
        'F1 Score': [r['f1'] for r in results.values()],
        'ROC AUC': [r['roc_auc'] for r in results.values()]
    })

    print("\n Model Comparison:")
    print(comparison.to_string(index=False))

    best_model_name = comparison.loc[comparison['F1 Score'].idxmax(), 'Model']
    print(f"\n Best Model (by F1 Score): {best_model_name}")

    fig = go.Figure()
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']

    for metric in metrics:
        fig.add_trace(go.Bar(
            name=metric,
            x=comparison['Model'],
            y=comparison[metric]
        ))

    fig.update_layout(
        title='Model Performance Comparison',
        xaxis_title='Model',
        yaxis_title='Score',
        barmode='group',
        height=500
    )
    fig.show()

    fig = go.Figure()

    for name, result in results.items():
        fpr, tpr, _ = roc_curve(y_test, result['probabilities'])
        fig.add_trace(go.Scatter(
            x=fpr, y=tpr,
            name=f"{name} (AUC = {result['roc_auc']:.3f})",
            mode='lines'
        ))

    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1],
        name='Random Classifier',
        mode='lines',
        line=dict(dash='dash', color='gray')
    ))

    fig.update_layout(
        title='ROC Curves',
        xaxis_title='False Positive Rate',
        yaxis_title='True Positive Rate',
        height=600
    )
    fig.show()

    return comparison, best_model_name

comparison, best_model = evaluate_models(results, y_test)

In [ ]:
def analyze_feature_importance(model, feature_names, top_n=20):
  """Analyze feature importance for the best model"""

  print("="*60)
  print("FEATURE IMPORTANCE ANALYSIS")
  print("="*60)

  if hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
  elif hasattr(model, 'coef_'):
    importances = model.coef_[0]
  else:
    print("Model doesn't support feature Importance")
    return

  feature_imp = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values('Importance', ascending=False).head(top_n)

  print(f"\nTop {top_n} Most Important Features:")
  print(feature_imp.to_string(index=False))

  fig = px.bar(feature_imp, x='Importance', y='Feature', orientation='h',
                 title=f'Top {top_n} Feature Importances')
  fig.update_layout(height=600)
  fig.show()

analyze_feature_importance(results[best_model]['model'], df.drop('Class', axis=1).columns)

In [ ]:
def save_model(model, scaler, filepath='fraud_detection_model'):
    """Save the trained model and scaler"""
    import pickle

    print("="*60)
    print("SAVING MODEL")
    print("="*60)

    with open(f'{filepath}_model.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f" Model saved to {filepath}_model.pkl")

    with open(f'{filepath}_scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    print(f" Scaler saved to {filepath}_scaler.pkl")

    try:
        from google.colab import files
        files.download(f'{filepath}_model.pkl')
        files.download(f'{filepath}_scaler.pkl')
        print(" Files ready for download!")
    except:
        print("ℹRun this in Colab to auto-download files")

save_model(results[best_model]['model'], scaler)

In [ ]:
total_test_samples = len(X_test)

if total_test_samples == 0:
    print("X_test is empty, cannot make predictions.")
elif total_test_samples == 1:
    num_samples_to_pick = 1
else:
    min_samples = 1
    max_samples = min(200, total_test_samples)

    if min_samples > max_samples:
        num_samples_to_pick = max_samples
    else:
        num_samples_to_pick = np.random.randint(min_samples, max_samples + 1)

    random_indices = np.random.choice(total_test_samples, size=num_samples_to_pick, replace=False)
    new_transaction_data = X_test[random_indices]

    scaled_data = scaler.transform(new_transaction_data)
    prediction = results[best_model]['model'].predict(scaled_data)
    probability = results[best_model]['model'].predict_proba(scaled_data)[:, 1]

    predictions_df = pd.DataFrame({
        'Original_X_test_Index': random_indices,
        'Prediction': ['FRAUD' if p == 1 else 'NORMAL' for p in prediction],
        'Fraud Probability': probability
    })

    print(f"Predicted on {num_samples_to_pick} randomly selected transactions:")
    print(predictions_df)

    print("\n--- Prediction Summary ---")
    print(predictions_df['Prediction'].value_counts())
